# Searching for self-similar blow-up profiles

*fluids · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/blowup_profile_search)

How singularities are hunted today: write $u = (T-t)^\lambda U\big(x/(T-t)^{1+\lambda}\big)$, turn the PDE into an equation for the profile $U$ with unknown exponent $\lambda$, and train a batch of candidate profiles, one per trial $\lambda$, as one tensor with Gauss–Newton. For inviscid Burgers the residual funnels collapse only at the exact $\lambda = 1/(2i+2)$, and the learned profiles match the classical $y = -U - U^{2i+3}$. The same search on the nonlocal Córdoba–Córdoba–Fontelos equation finds its stable exponent $\lambda \approx 1.18$.

**Vector form**

$$u_t + u\,u_x = 0,\qquad u = \tau^{\lambda}\,U(y),\ \ y = \frac{x}{\tau^{1+\lambda}},\ \ \tau = T - t\ \ \Longrightarrow\ \ -\lambda U + \big((1+\lambda)y + U\big)U' = 0,\qquad \theta_t - (H\theta)\,\theta_x = 0\ \Longrightarrow\ \Omega + \big((1+\lambda)y - V\big)\Omega' - \Omega\,H\Omega = 0,\ \ V = \textstyle\int_0^y H\Omega$$

**Index (tensor) form**

$$R_{bn} = -\lambda_b U_{bn} + \big((1+\lambda_b)\,y_{bn} + U_{bn}\big)\,U'_{bn},\qquad U_{bn} = y_{bn}\big(-r_n + \phi_{nk}\,c_{bk}\big),\qquad c_b \leftarrow c_b - \big(J_b^{\mathsf T}J_b + \mu_b D_b\big)^{-1}J_b^{\mathsf T}R_b$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\lambda_b$ | `lams = linspace(0.1, 0.7, n_lambda)` | (B,) | the batch: one trial exponent per candidate profile, all trained at once |
| $r_n$ | `chebyshev_points(N_COL)` | (N,) | compactified coordinate $r = (1 + y^2/L^2)^{-1/(2(1+\lambda))}$: $r = 1$ at $y = 0$, $r \to 0$ at $/y/ \to \infty$ |
| $U = y\,P(r)$ | `burgers_profile(c, lam, y)` | (N,) | odd, $U'(0) = -1$ and $U \sim y^{\lambda/(1+\lambda)}$ far away are built in; $P$ is a Chebyshev series in $r$ |
| $R_{bn}$ | `burgers_batch(C, lam, r, level)` | (B, M) | profile-equation residual at the collocation points, plus $U(2) = -1$ and the smoothness constraints |
| $J_{bmk} = \partial R_{bm}/\partial c_{bk}$ | `einsum('bnk,bk->bn', phi, C)` | (B, M, K) | $R$ is quadratic in $c$, so the Jacobian is exact (checked against autograd in the tests) |
| $(J^{\mathsf T}J + \mu D)^{-1}J^{\mathsf T}R$ | `linalg.lstsq(cat([J, sqrt(mu*D)]), -R)` | (B, K) | Levenberg–Marquardt (damped Gauss–Newton) step for every member of the batch in one batched solve |
| $c_3 = 0,\ c_5 = 0$ | `taylor_at_origin(c, lam)` | (2,) | Taylor coefficients of $U = -y + c_3y^3 + c_5y^5 + \dots$; forcing them to zero removes the funnels at $\lambda = 1/2$ and $1/4$ |
| $H\big[\operatorname{Im}(1-iy)^{-s}\big]$ | `-((1 - 1j*y)**(-s)).real` | (N, K) | exact Hilbert transform of the CCF basis: $(1-iy)^{-s}$ is analytic in the upper half plane |
| $\lambda^\ast$ | `golden_section(fit, lo, hi, C, n_iter, n_fit)` | (B,) | pins down the bottom of each funnel; the profile is retrained at every trial $\lambda$ |

Companion to fluids/navier_stokes_blowup, which shows the 1983 picture: run the Taylor–Green vortex forward and watch the vorticity grow until the grid gives out. The modern route skips the time evolution. It assumes the singularity is self-similar and searches for the profile directly. Plugging $u = \tau^\lambda U(y)$ into $u_t + uu_x = 0$ gives $-\lambda U + ((1+\lambda)y + U)U' = 0$. Solved as a linear ODE for $y(U)$ this is $y = -U - C\,U^{1+1/\lambda}$, which is smooth only when $1 + 1/\lambda$ is an odd integer. So $\lambda = 1/(2i+2)$, with exact profiles $y = -U - U^{2i+3}$ (normalised by $U'(0) = -1$ and $U(-2) = 1$). These are classical results (Eggers & Fontelos, Singularities, 2015). $\lambda = 1/2$ is the stable, generic blow-up; $1/4, 1/6, \dots$ need fine-tuned data. Here 61 candidate profiles are trained at once at fixed $\lambda$. The best residual against $\lambda$ (the funnel) falls to round-off ($\sim10^{-15}$) at $\lambda = 1/2$ and to $\sim10^{-11}$ at $1/4$. In between it is far larger ($\sim10^{7}$ times larger at $\lambda = 0.37$), because there only profiles with a non-smooth $|y|^{1+1/\lambda}$ term exist. Imposing $U'''(0) = 0$ removes the $1/2$ funnel, and the search then finds $1/4$. Imposing $U^{(5)}(0) = 0$ as well should single out $1/6$. At this resolution, though, the funnel near $1/6$ is too flat to pin it down: the competing profiles differ only at order $y^{7}$, and the shallow minimum lands near $\lambda \approx 0.19$. Note that the residual cannot tell stable from unstable. That is a property of the time evolution, and the unstable profiles are reached here only through the extra constraints. The profile is a spectral expansion with the inductive biases the research uses, a compactified coordinate and output shaping. Research papers use physics-informed neural networks (PINNs) with the same biases. Gauss–Newton is the optimiser of Wang et al. (2025). Gradient steps on $\lambda$ stall at the cusp of each funnel, so a batched golden-section search pins $\lambda$ down. The CCF equation $\theta_t - (H\theta)\theta_x = 0$ (a 1D model of the surface quasi-geostrophic equation; $H$ is the Hilbert transform) is solved in vorticity form, $\omega = \theta_x = \Omega(y)/\tau$. $\Omega$ is expanded in $\operatorname{Im}(1-iy)^{-s}$, $s = j/(1+\lambda) + m$, whose Hilbert transforms are exact. The default run finds $\lambda \approx 1.18$ (residual $\sim10^{-4}$); longer runs with a larger basis settle at $1.1808$ (residual $2\times10^{-6}$). This agrees with the ground-state exponent $\approx 1.18$ of Eggers & Fontelos (Nonlinearity, 2020). Wang, Lai, Gómez-Serrano & Buckmaster (PRL, 2023) found the first unstable CCF profile with PINNs, $\lambda_1 \approx 0.6057$. Wang et al. (2025, arXiv 2509.14185) found a second, $\lambda_2 \approx 0.4713$, and reached residuals near $10^{-13}$, close to double precision. They also found new unstable families for the porous-media and Boussinesq equations, which are numerical candidates. The unstable profiles have sharp gradients that this one-minute CPU search does not resolve; it shows only the stable funnel. Status: the Burgers profiles are exact theorems. CCF blow-up was proved by Córdoba, Córdoba & Fontelos (Annals, 2005). Profiles found by neural search are numerical candidates until a computer-assisted proof turns them into theorems, as Chen & Hou (2022) did for the Hou–Luo axisymmetric Euler scenario with a boundary. The 3D incompressible Navier–Stokes Millennium Problem remains open, and nothing here bears on it directly. Try a coarser $\lambda$ grid or fewer Gauss–Newton steps, and watch the funnels blur.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/blowup_profile_search/main.py
"""Main loop: λ grid → train a batch of profiles (the funnels) → pin down λ → blow-up of the found profile → CCF."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve

EXACT = [0.5, 0.25, 1 / 6]  # λ = 1/(2i + 2) of the smooth Burgers profiles, i = 0, 1, 2
CCF_REFERENCE = {  # published exponents for u_t − (Hu) u_x = 0 (same λ convention as here)
    "stable ≈ 1.18 (EF 2020)": 1.18,
    "unstable 0.6057 (Wang 2023)": 0.6057,
    "unstable 0.4713 (Wang 2025)": 0.47132,
}


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    f64 = dict(dtype=torch.float64, device=dev)
    n_lam = 13 if fast_mode() else params["n_lambda"]
    iters = 8 if fast_mode() else params["gn_steps"]
    lams = torch.linspace(0.1, 0.7, n_lam, **f64)  # the batch: one candidate profile per trial λ
    spacing = (lams[1] - lams[0]).item()
    timing, t0 = {}, time.perf_counter()

    # 1. Funnels: every λ of the grid trained at once, for the three smoothness levels.
    funnels, paths, found = [], [], []
    for level in range(3):
        path, hist = solve.burgers_funnel_history(lams, level, iters)
        funnels.append(hist)
        paths.append(path)
    print(f"Burgers: {n_lam} profiles (λ = 0.1 … 0.7) × 3 smoothness levels, {iters} Gauss–Newton steps each")
    for level, name in enumerate(["no constraint", "U‴(0) = 0", "U‴(0) = U⁽⁵⁾(0) = 0"]):
        res = funnels[level][-1]
        minima = solve.local_minima(res)
        start = lams[minima[0]] if len(minima) else lams[res.argmin()]
        out = solve.burgers_search(
            start.reshape(1), level, width=1.5 * spacing, n_golden=6 if fast_mode() else 20, n_fit=16
        )
        lam = out["lam"][0].item()
        i = max(0, round(1 / (2 * lam)) - 1)
        y = torch.linspace(-5, 5, 401, **f64)
        U, _ = equation.burgers_profile(out["coef"][0], out["lam"][0], y)
        sup = (U - equation.burgers_exact(y, i)[0]).abs().max().item()
        found.append({"lam": lam, "coef": out["coef"][0], "res": out["res"][0].item(), "hist": out["lam_hist"][:, 0]})
        print(
            f"  level {level} ({name:<20}): deepest funnel at λ ≈ {start.item():.3f};  search → λ = {lam:.6f} "
            f"(exact family member 1/{2 * i + 2} = {1 / (2 * i + 2):.6f}, error {abs(lam - 1 / (2 * i + 2)):.1e}), "
            f"‖R‖ = {out['res'][0]:.1e}, sup|U − U_exact| on |y| ≤ 5 = {sup:.1e}"
        )
    timing["Burgers"] = time.perf_counter() - t0
    j = int((lams - 0.37).abs().argmin())
    k = int((lams - 0.5).abs().argmin())
    print(
        f"  funnel contrast: ‖R‖(λ = {lams[j]:.2f}) / ‖R‖(λ = {lams[k]:.2f}) = "
        f"{funnels[0][-1][j] / funnels[0][-1][k]:.1e} (smooth profiles exist only at λ = 1/(2i+2))"
    )

    # 2. The blow-up itself: u(x, t) = τ^λ U(x / τ^{1+λ}) with the found stable profile.
    lam0, c0 = torch.tensor(found[0]["lam"], **f64), found[0]["coef"]
    taus = torch.logspace(0, -3, 12 if fast_mode() else 30, **f64)  # τ = T − t
    x = torch.linspace(-1, 1, 801, **f64)
    u = torch.stack([tau**lam0 * equation.burgers_profile(c0, lam0, x / tau ** (1 + lam0))[0] for tau in taus])
    slope = equation.burgers_profile(c0, lam0, torch.zeros(1, **f64))[1].abs() * taus**-1.0  # |U′(0)| / τ
    print(
        f"blow-up: max|u_x| · (T − t) = {(slope * taus).min():.6f} … {(slope * taus).max():.6f} for T − t = 1 … 1e-3 "
        "(the gradient diverges like 1/(T − t); u itself stays bounded, |u| ≤ τ^λ max|U| on the window)"
    )

    # 3. CCF: funnel and search.
    clams = torch.linspace(0.45, 1.6, 7 if fast_mode() else 24, **f64)
    _, ccf_hist = solve.ccf_funnel_history(clams, 6 if fast_mode() else 30)
    t0 = time.perf_counter()
    ccf = solve.ccf_search(
        torch.tensor([params["ccf_lambda0"]], **f64),
        n_golden=4 if fast_mode() else 14,
        n_fit=8 if fast_mode() else 30,
        polish=2 if fast_mode() else 4,
        warm=10 if fast_mode() else 60,
    )
    timing["CCF"] = time.perf_counter() - t0
    lam_c = ccf["lam"][0].item()
    print(
        f"CCF (θ_t − (Hθ)θ_x = 0): search from λ0 = {params['ccf_lambda0']:.2f} → λ = {lam_c:.5f}, "
        f"‖R‖ = {ccf['res'][0]:.1e} ({solve.CCF_COL} collocation points, {ccf['coef'].shape[1]} basis functions)"
    )
    print(
        "  published: stable λ ≈ 1.18 (Eggers & Fontelos 2020, similarity exponent of the ground state); unstable "
        "λ₁ ≈ 0.6057 and λ₂ ≈ 0.4713 (Wang et al. 2023, 2025). This reduced CPU search resolves only the stable "
        "branch: the unstable profiles have sharp gradients that need far more resolution."
    )
    print("compute time: " + ", ".join(f"{k} {v:.1f} s" for k, v in timing.items()))
    return plot.animate(
        lams,
        funnels,
        paths,
        found,
        EXACT,
        taus,
        x,
        u,
        slope,
        clams,
        ccf_hist[-1],
        ccf,
        CCF_REFERENCE,
        out_dir / "blowup_profile_search.gif",
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/blowup_profile_search/equation.py
r"""Physics → tensor translation: self-similar blow-up profiles.

Ansatz u(x, t) = τ^λ U(y), τ = T − t, y = x / τ^{1+λ}. Then u_t = τ^{λ−1}[−λU + (1+λ) y U′] and u u_x = τ^{λ−1} U U′,
so inviscid Burgers u_t + u u_x = 0 becomes the profile equation

    −λ U + ((1 + λ) y + U) U′ = 0,

with λ unknown. Solving it as a linear ODE for y(U) gives y = −U − C U^{1 + 1/λ}: the profile is smooth (C^∞) only
when 1 + 1/λ is an odd integer 2i + 3, i.e. λ = 1/(2i + 2); with U′(0) = −1 and C = 1, y = −U − U^{2i+3}.

CCF, θ_t − (Hθ) θ_x = 0 (Hilbert transform H, H[cos] = sin). With ω = θ_x = Ω(y)/τ the velocity becomes
V(y) = ∫₀^y HΩ, and the vorticity profile obeys

    Ω + ((1 + λ) y − V) Ω′ − Ω HΩ = 0.

For F(y) = (1 − iy)^{−s} (analytic in the upper half plane): H[Re F] = Im F, so H[Im F] = −Re F.
"""

import torch

from . import tensors


# ---------------------------------------------------------------- Burgers
def burgers_residual(y: torch.Tensor, U: torch.Tensor, dU: torch.Tensor, lam) -> torch.Tensor:
    """R = −λU + ((1+λ)y + U) U′, elementwise (any shape)."""
    return -lam * U + ((1 + lam) * y + U) * dU


def burgers_exact(y: torch.Tensor, i: int) -> tuple[torch.Tensor, torch.Tensor]:
    """The exact smooth profile of order i (λ = 1/(2i+2)): solve y = −U − U^n, n = 2i + 3, by Newton on every point
    at once. Returns U and U′ = −1/(1 + n U^{n−1})."""
    n = 2 * i + 3
    U = -y / (1 + y.abs()) ** (1 - 1 / n)  # right size near 0 and at infinity
    for _ in range(60):
        U = U - (U + U**n + y) / (1 + n * U ** (n - 1))
    return U, -1 / (1 + n * U ** (n - 1))


def compactify(y: torch.Tensor, lam: torch.Tensor, L: float = tensors.L_BURGERS) -> torch.Tensor:
    """r = (1 + y²/L²)^{−g}, g = 1/(2(1+λ))."""
    return (1 + (y / L) ** 2) ** (-1 / (2 * (1 + lam)))


def burgers_profile(
    c: torch.Tensor, lam: torch.Tensor, y: torch.Tensor, L: float = tensors.L_BURGERS
) -> tuple[torch.Tensor, torch.Tensor]:
    """U(y) = y P(r) with P(r) = −r + Σ c_k φ_k(r), and U′ = P + y P′(r) dr/dy.   c (3K,), y (N,) → (N,), (N,)"""
    g = 1 / (2 * (1 + lam))
    q = (y / L) ** 2
    r = (1 + q) ** (-g)
    phi, dphi = tensors.burgers_features(r, g, c.shape[-1] // tensors.N_FAMILIES)
    P = -r + phi @ c
    dP = -1 + dphi @ c
    drdy = -2 * g * y * r / (L**2 * (1 + q))
    return y * P, P + y * dP * drdy


def taylor_at_origin(c: torch.Tensor, lam: torch.Tensor, L: float = tensors.L_BURGERS) -> torch.Tensor:
    """Taylor coefficients (c₃, c₅) of U(y) = −y + c₃y³ + c₅y⁵ + …, from the series of y P(r(y)) at y = 0:
    c₃ = −g P′(1)/L², c₅ = g((g + 1)P′(1) + g P″(1))/(2L⁴). A profile with c₃ = 0 cannot sit at λ = 1/2 (the y³
    balance of the profile equation reads c₃(2λ − 1) = 0), one with c₃ = c₅ = 0 not at 1/4 either."""
    g = 1 / (2 * (1 + lam))
    d = tensors.feature_derivatives_at_one(c.shape[-1] // tensors.N_FAMILIES, g.reshape(1))[0]  # (2, 3K)
    p1, p2 = d[0] @ c - 1, d[1] @ c  # P = −r + …
    return torch.stack([-p1 * g / L**2, g * ((g + 1) * p1 + g * p2) / (2 * L**4)])


def max_slope(tau: torch.Tensor, lam: float) -> torch.Tensor:
    """max |u_x| = τ^λ |U′(0)| / τ^{1+λ} = 1/τ for the normalised profile U′(0) = −1: the same for every λ."""
    return tau**lam * 1.0 / tau ** (1 + lam)


# ---------------------------------------------------------------- Hilbert transform and CCF
def hilbert_periodic(f: torch.Tensor) -> torch.Tensor:
    """Periodic Hilbert transform along the last axis by FFT: Ĥf(k) = −i sgn(k) f̂(k), so H[cos] = sin."""
    n = f.shape[-1]
    k = torch.fft.fftfreq(n, 1 / n, dtype=f.dtype, device=f.device)
    return torch.fft.ifft(-1j * torch.sign(k) * torch.fft.fft(f)).real


def hilbert_line(f: torch.Tensor) -> torch.Tensor:
    """Hilbert transform on the real line for samples on the circle grid y = tan(φ/2) (tensors.circle_grid): the
    periodic transform in φ, minus its value at φ = π (y = ±∞), so that Hf → 0 at infinity."""
    n = f.shape[-1]
    k = torch.fft.fftfreq(n, 1 / n, dtype=f.dtype, device=f.device)
    f_hat = torch.fft.fft(f)
    shift = torch.exp(1j * torch.pi * k * (2 - 1 / n))  # e^{ik(π − φ₀)}, the grid starts at φ₀ = −π + π/n
    at_infinity = (-1j * torch.sign(k) * f_hat * shift).sum().real / n
    return hilbert_periodic(f) - at_infinity


def ccf_basis_pair(y: torch.Tensor, s: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Odd basis functions Im (1 − iy)^{−s} and their exact Hilbert transforms −Re (1 − iy)^{−s}: (N, K) each."""
    F = (1 - 1j * y)[:, None] ** (-s)
    return F.imag, -F.real


def ccf_fields(a: torch.Tensor, lam: torch.Tensor, y: torch.Tensor, n_j: int, n_m: int) -> dict:
    """Ω = Σ a_k Im F_k, HΩ, V = ∫₀^y HΩ and Ω′ for F_k = (1 − iy)^{−s_k}.   a (K,), y (N,) → dict of (N,)"""
    s = tensors.ccf_exponents(1 / (1 + lam), n_j, n_m)  # (K,)
    z = (1 - 1j * y)[:, None]  # (N, 1)
    F = z ** (-s)  # (N, K)
    one = (s - 1).abs() < 1e-12  # s = 1: ∫ 1/(1+y²) = arctan y
    V_basis = torch.where(one, -torch.atan(y)[:, None], (z ** (1 - s)).imag / torch.where(one, 1.0, 1 - s))
    dF = (1j * s * z ** (-s - 1)).imag  # d/dy Im (1 − iy)^{−s}
    return {"omega": F.imag @ a, "h_omega": -F.real @ a, "V": V_basis @ a, "d_omega": dF @ a, "d_omega0": s @ a}


def ccf_residual(a: torch.Tensor, lam: torch.Tensor, y: torch.Tensor, n_j: int, n_m: int) -> torch.Tensor:
    """Ω + ((1+λ)y − V)Ω′ − Ω HΩ at the collocation points, scaled by (1 + y²)^{β/2} so the far field counts as
    much as the core, plus the normalisation Ω′(0) = −1 (weight 10): (N + 1,)."""
    f = ccf_fields(a, lam, y, n_j, n_m)
    R = f["omega"] + ((1 + lam) * y - f["V"]) * f["d_omega"] - f["omega"] * f["h_omega"]
    R = R * (1 + y**2) ** (0.5 / (1 + lam))
    return torch.cat([R, 10 * (f["d_omega0"] + 1)[None]])


# ---------------------------------------------------------------- batched residuals with their Jacobians
def burgers_batch(C: torch.Tensor, lam: torch.Tensor, r: torch.Tensor, level: int, need_jac: bool = True) -> tuple:
    """Residual vectors R (B, M) of a batch of candidate profiles C (B, 3K) at exponents λ (B,), and their exact
    Jacobians J = ∂R/∂C (B, M, 3K). U = y(−r + φ·c) is linear in c, so R is quadratic and J has a closed form.
    Rows: the profile equation at the collocation points r (N,) (weighted by r ≈ 1/|U| far away); U(2) = −1, which
    fixes C = 1 in y = −U − C U^{2i+3}; and for level ≥ 1 the constraints c₃ = 0 (level 2: also c₅ = 0).
    With need_jac=False only R is computed (J is None)."""
    L = tensors.L_BURGERS
    n_modes = C.shape[-1] // tensors.N_FAMILIES
    g = (1 / (2 * (1 + lam)))[:, None]  # (B, 1)
    y = L * (r ** (-1 / g) - 1).clamp_min(0).sqrt()  # (B, N): the points follow λ
    phi, dphi = tensors.burgers_features(r, g, n_modes)  # (B, N, 3K)
    P = -r + torch.einsum("bnk,bk->bn", phi, C)
    dP = -1 + torch.einsum("bnk,bk->bn", dphi, C)
    drdy = -2 * g * y * r / (L**2 + y**2)
    U, dU = y * P, P + y * dP * drdy
    R = burgers_residual(y, U, dU, lam[:, None]) * r
    if need_jac:
        JU = y[..., None] * phi  # (B, N, 3K)
        JdU = phi + (y * drdy)[..., None] * dphi
        J = ((-lam[:, None] + dU)[..., None] * JU + ((1 + lam[:, None]) * y + U)[..., None] * JdU) * r[:, None]
    r2 = (1 + (2 / L) ** 2) ** (-g)  # (B, 1) the point y = 2
    phi2, _ = tensors.burgers_features(r2, g, n_modes)  # (B, 1, 3K)
    rows = [R, 20 * (-r2 + torch.einsum("bnk,bk->bn", phi2, C) + 0.5)]
    jac = [J, 20 * phi2] if need_jac else []
    if level >= 1:  # c₃L² = −g p₁ and c₅L⁴ = g((g+1)p₁ + g p₂)/2 with p_m = P^{(m)}(1)
        d = tensors.feature_derivatives_at_one(n_modes, g[:, 0])  # (B, 2, 3K)
        p1, p2 = (d[:, 0] * C).sum(-1) - 1, (d[:, 1] * C).sum(-1)
        rows.append(10 * (-g[:, 0] * p1)[:, None])
        jac.append(10 * (-g * d[:, 0])[:, None])
        if level >= 2:
            rows.append(10 * (g[:, 0] * ((g[:, 0] + 1) * p1 + g[:, 0] * p2) / 2)[:, None])
            jac.append(10 * (g * ((g + 1) * d[:, 0] + g * d[:, 1]) / 2)[:, None])
    return torch.cat(rows, 1), torch.cat(jac, 1) if need_jac else None


def ccf_batch(A: torch.Tensor, lam: torch.Tensor, y: torch.Tensor, n_j: int, n_m: int, need_jac: bool = True) -> tuple:
    """CCF residual vectors R (B, N + 1) and exact Jacobians ∂R/∂a (B, N + 1, K) for a batch of coefficient vectors
    A (B, K) at exponents λ (B,). Ω, HΩ, V and Ω′ are all linear in a, so R is quadratic in a."""
    s = torch.stack([tensors.ccf_exponents(1 / (1 + lv), n_j, n_m) for lv in lam])[:, None, :]  # (B, 1, K)
    z = (1 - 1j * y)[None, :, None]  # (1, N, 1)
    F = z ** (-s)  # (B, N, K)
    one = (s - 1).abs() < 1e-12  # s = 1: ∫ 1/(1+y²) dy = arctan y
    Om, Hm = F.imag, -F.real
    Vm = torch.where(one, -torch.atan(y)[None, :, None], (z ** (1 - s)).imag / torch.where(one, 1.0, 1 - s))
    Dm = (1j * s * z ** (-s - 1)).imag
    om, h, v, d = (torch.einsum("bnk,bk->bn", M, A) for M in (Om, Hm, Vm, Dm))
    c = (1 + lam[:, None]) * y - v  # characteristic speed (B, N)
    w = (1 + y**2) ** (0.5 / (1 + lam[:, None]))  # far-field weight (B, N)
    R = (om + c * d - om * h) * w
    s0 = s[:, 0, :]  # Ω′(0) = Σ s_k a_k
    R = torch.cat([R, 10 * ((s0 * A).sum(-1) + 1)[:, None]], 1)
    if not need_jac:
        return R, None
    J = (Om + c[..., None] * Dm - d[..., None] * Vm - h[..., None] * Om - om[..., None] * Hm) * w[..., None]
    return R, torch.cat([J, 10 * s0[:, None, :]], 1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/blowup_profile_search/tensors.py
"""Build the concrete tensors: collocation points, the spectral bases that represent a profile, and the batch of
initial exponents λ that the search starts from.

Burgers. A profile is odd, U(0) = 0, U′(0) = −1, and grows like |y|^{λ/(1+λ)} far away. Both facts are built into the
representation U(y) = y P(r), with the compactified coordinate r = (1 + y²/L²)^{−g} ∈ (0, 1], g = 1/(2(1+λ)): r = 1 at
y = 0 and r → 0 as |y| → ∞. P is a Chebyshev series in x = 2r − 1 with P(1) = −1 (so U′(0) = −1) and P(0) = 0 (so
U ~ y^{λ/(1+λ)}). The 2025 paper calls this kind of choice the inductive bias of the network: coordinate transforms
and output shaping.

CCF. The vorticity profile Ω(y) is odd and decays like |y|^{−β}, β = 1/(1+λ). It is expanded in the boundary values of
(1 − iy)^{−s}, which are analytic in the upper half plane, so their Hilbert transforms are known exactly. The exponents
are s = jβ + m (j = 0 … J, m = 0 … M, s ≠ 0): the powers that appear in the far-field expansion.
"""

import math

import torch

L_BURGERS = 0.3  # length scale of the compactification; keeps the complex singularities of U far from r ∈ [0, 1]


def chebyshev_points(n: int, device: torch.device) -> torch.Tensor:
    """Chebyshev (first kind) points mapped to r ∈ (0, 1): (n,). Dense near r = 0 (far field) and r = 1 (y = 0)."""
    k = torch.arange(n, dtype=torch.float64, device=device)
    return 0.5 * (1 - torch.cos(math.pi * (k + 0.5) / n))


def chebyshev(x: torch.Tensor, n_modes: int) -> tuple[torch.Tensor, torch.Tensor]:
    """T_k(x) and T_k′(x) for k = 1 … K by the three-term recurrence: (…, K) each."""
    t, dt = [torch.ones_like(x), x], [torch.zeros_like(x), torch.ones_like(x)]
    for _ in range(n_modes - 1):
        t.append(2 * x * t[-1] - t[-2])
        dt.append(2 * t[-2] + 2 * x * dt[-1] - dt[-2])
    return torch.stack(t[1 : n_modes + 1], -1), torch.stack(dt[1 : n_modes + 1], -1)


def burgers_basis(r: torch.Tensor, n_modes: int) -> tuple[torch.Tensor, torch.Tensor]:
    """b_k(r) = T_k(x) − (1+x)/2 − (−1)^k (1−x)/2, x = 2r − 1, which vanish at r = 0 and r = 1, and db_k/dr.
    r (N,) → (N, K), (N, K)."""
    x = 2 * r - 1
    t, dt = chebyshev(x, n_modes)
    sign = (-1.0) ** torch.arange(1, n_modes + 1, dtype=r.dtype, device=r.device)  # T_k(−1) = (−1)^k
    b = t - (1 + x[..., None]) / 2 - sign * (1 - x[..., None]) / 2
    db = 2 * dt - 1 + sign
    return b, db


def basis_derivatives_at_one(n_modes: int, device: torch.device) -> torch.Tensor:
    """d^m b_k / dr^m at r = 1 (y = 0) for m = 1, 2, 3: (3, K). Uses T_k^{(m)}(1) = Π_{j<m} (k² − j²)/(2j + 1)."""
    k2 = torch.arange(1, n_modes + 1, dtype=torch.float64, device=device) ** 2
    sign = (-1.0) ** torch.arange(1, n_modes + 1, dtype=torch.float64, device=device)
    d1 = 2 * k2 - 1 + sign
    d2 = 4 * k2 * (k2 - 1) / 3
    d3 = 8 * k2 * (k2 - 1) * (k2 - 4) / 15
    return torch.stack([d1, d2, d3])


def circle_grid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """φ_j = −π + 2π(j + ½)/n and y_j = tan(φ_j/2): the real line wrapped onto the circle (Möbius map), where the
    Hilbert transform of the line becomes the periodic Hilbert transform. (n,), (n,)"""
    phi = -math.pi + 2 * math.pi * (torch.arange(n, dtype=torch.float64, device=device) + 0.5) / n
    return phi, torch.tan(phi / 2)


def ccf_points(n: int, device: torch.device) -> torch.Tensor:
    """Collocation points y = tan(φ/2) for φ ∈ (0, 0.999π): y from ≈ 0 to ≈ 1300 (the profile is odd): (n,)."""
    phi = (torch.arange(n, dtype=torch.float64, device=device) + 0.5) / n * math.pi * 0.999
    return torch.tan(phi / 2)


def ccf_exponents(beta: torch.Tensor, n_j: int, n_m: int) -> torch.Tensor:
    """s = jβ + m for j = 0 … J, m = 0 … M without s = 0: (K,) with K = (J+1)(M+1) − 1."""
    j = torch.arange(n_j + 1, dtype=beta.dtype, device=beta.device)[:, None]
    m = torch.arange(n_m + 1, dtype=beta.dtype, device=beta.device)[None]
    return (j * beta + m).flatten()[1:]


def ccf_initial_coefficients(lam: torch.Tensor, n_j: int, n_m: int) -> torch.Tensor:
    """Start every member of the batch from Ω = −(1+λ) Im (1 − iy)^{−β} (so Ω′(0) = −1): (B, K)."""
    a = torch.zeros(lam.shape[0], (n_j + 1) * (n_m + 1) - 1, dtype=lam.dtype, device=lam.device)
    a[:, n_m] = -(1 + lam)  # the (j, m) = (1, 0) column, s = β
    return a


N_FAMILIES = 1  # powers w^j (j < N_FAMILIES) of w = 1/(1 + y²/L²) = r^{1/g}; extra families slowed convergence here


def burgers_features(r: torch.Tensor, g: torch.Tensor, n_modes: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Features w^j b_k(r) (j = 0 … 2, k = 1 … K) and their r-derivatives. The far field of U contains powers of
    y^{−2g} ∝ r and of y^{−2} ∝ w = r^{1/g}; both are covered, and w = 1/(1 + y²/L²) is analytic at y = 0 too.
    r and g broadcast against each other (e.g. r (N,), g (B, 1) → (B, N)) → features (…, 3K), d/dr (…, 3K)."""
    g = torch.as_tensor(g, dtype=r.dtype, device=r.device)
    r, g = torch.broadcast_tensors(r, g)
    b, db = burgers_basis(r, n_modes)  # (…, K)
    j = torch.arange(N_FAMILIES, dtype=r.dtype, device=r.device)
    wj = (r ** (1 / g))[..., None, None] ** j  # (…, 1, 3): w^j
    phi = (wj * b[..., None]).flatten(-2)  # (…, K·3), index 3k + j
    dwj = j * wj / (g * r)[..., None, None]  # d w^j / dr = (j/g) w^j / r
    dphi = (dwj * b[..., None] + wj * db[..., None]).flatten(-2)
    return phi, dphi


def feature_derivatives_at_one(n_modes: int, g: torch.Tensor) -> torch.Tensor:
    """P′(1) and P″(1) contributions of every feature (b_k(1) = 0, w(1) = 1): (B, 2, 3K) for g (B,)."""
    d = basis_derivatives_at_one(n_modes, g.device)  # (3, K)
    j = torch.arange(N_FAMILIES, dtype=g.dtype, device=g.device)
    d1 = d[0][None, :, None].expand(len(g), -1, N_FAMILIES)  # (B, K, 3)
    d2 = d[1][None, :, None] + 2 * (j / g[:, None, None]) * d[0][None, :, None]
    return torch.stack([d1.flatten(1), d2.flatten(1)], 1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/blowup_profile_search/solve.py
"""The search: a batch of candidate profiles, one per trial exponent λ, trained as one tensor.

Profiles: Gauss–Newton (Levenberg–Marquardt) on the expansion coefficients, the second-order optimiser the 2025 paper
uses to reach high precision; every member of the batch takes its own step in one batched least-squares solve.
Exponent: the best residual at fixed λ, the funnel, has a cusp at every admissible λ, so plain gradient steps on λ
stall next to it. A batch scan over a grid of λ finds the funnels, and a golden-section search, with the profile
re-trained at every trial λ, pins each one down. For CCF a final joint Gauss–Newton step trains profile and λ
together, with the Jacobian (including ∂R/∂λ) from autograd.
"""

import torch

from problems._common.progress import steps

from . import equation, tensors


def gauss_newton(residual, C: torch.Tensor, lam: torch.Tensor, iters: int, keep_path: bool = False) -> tuple:
    """Levenberg–Marquardt on every row of C (B, K) at fixed λ (B,). residual(C, λ, need_jac) → R (B, M) and
    J = ∂R/∂C (B, M, K) (None when need_jac is False).
    Each step solves min ‖J d + R‖² + μ‖D d‖² (D² = diag JᵀJ) by least squares; μ shrinks after a successful step
    and grows (and the step is retried once) after a failed one, per member. Returns C, ‖R‖ (B,) and the ‖R‖
    history (iters + 1, B); with keep_path also the coefficients after every step (iters + 1, B, K)."""
    B, K = C.shape
    mu = torch.full((B,), 1e-3, dtype=C.dtype, device=C.device)
    R, J = residual(C, lam, True)
    hist, path = [R.norm(dim=1)], [C]
    for _ in range(iters):
        D = torch.einsum("bmk,bmk->bk", J, J)  # diag JᵀJ, repeated m summed
        rhs = torch.cat([-R, torch.zeros_like(C)], 1)[..., None]
        todo = torch.ones(B, dtype=torch.bool, device=C.device)
        for _ in range(2):  # one retry with a larger μ for the members whose step failed
            A = torch.cat([J, torch.diag_embed((mu[:, None] * D + 1e-14).sqrt())], 1)  # (B, M + K, K)
            d = torch.linalg.lstsq(A, rhs).solution[..., 0]
            R_new, _ = residual(C + d, lam, False)
            better = todo & (R_new.norm(dim=1) < R.norm(dim=1))
            C = torch.where(better[:, None], C + d, C)
            mu = torch.where(better, (mu / 3).clamp_min(1e-14), torch.where(todo, mu * 4, mu))
            todo = todo & ~better
            if not todo.any():
                break
        R, J = residual(C, lam, True)
        hist.append(R.norm(dim=1))
        if keep_path:
            path.append(C)
    if keep_path:
        return C, hist[-1], torch.stack(hist), torch.stack(path)
    return C, hist[-1], torch.stack(hist)


def golden_section(residual, lo: torch.Tensor, hi: torch.Tensor, C: torch.Tensor, n_iter: int, n_fit: int, label: str):
    """Pin down λ in each bracket [lo, hi] (B,) by golden-section search on the funnel: at every trial λ the profiles
    are re-trained for n_fit Gauss–Newton steps (warm start). Every member has its own bracket; all are updated at
    once with torch.where. Returns the best λ, its coefficients, ‖R‖ and the λ history (n_iter + 1, B)."""
    phi = (5**0.5 - 1) / 2
    x1, x2 = hi - phi * (hi - lo), lo + phi * (hi - lo)
    C1, f1, _ = gauss_newton(residual, C, x1, n_fit)
    C2, f2, _ = gauss_newton(residual, C, x2, n_fit)
    hist = [torch.where(f1 < f2, x1, x2)]
    for _ in steps(n_iter, label):
        left = f1 < f2  # the minimum lies in [lo, x2]: x1 becomes the new x2
        lo, hi = torch.where(left, lo, x1), torch.where(left, x2, hi)
        x_new = torch.where(left, hi - phi * (hi - lo), lo + phi * (hi - lo))
        x_keep, f_keep = torch.where(left, x1, x2), torch.where(left, f1, f2)
        C_keep = torch.where(left[:, None], C1, C2)
        C_new, f_new, _ = gauss_newton(residual, C_keep, x_new, n_fit)
        x1, x2 = torch.where(left, x_new, x_keep), torch.where(left, x_keep, x_new)
        f1, f2 = torch.where(left, f_new, f_keep), torch.where(left, f_keep, f_new)
        C1, C2 = torch.where(left[:, None], C_new, C_keep), torch.where(left[:, None], C_keep, C_new)
        hist.append(torch.where(f1 < f2, x1, x2))
    best = f1 < f2
    return torch.where(best, x1, x2), torch.where(best[:, None], C1, C2), torch.where(best, f1, f2), torch.stack(hist)


def local_minima(res: torch.Tensor) -> torch.Tensor:
    """Indices of the interior local minima of a funnel res (B,), deepest first."""
    i = torch.nonzero((res[1:-1] < res[:-2]) & (res[1:-1] < res[2:])).flatten() + 1
    return i[torch.argsort(res[i])]


# ---------------------------------------------------------------- Burgers
N_MODES, N_COL = 64, 220


def burgers_residual_fn(level: int, device: torch.device):
    r = tensors.chebyshev_points(N_COL, device)
    return lambda C, lam, need_jac=True: equation.burgers_batch(C, lam, r, level, need_jac)


def burgers_funnel_history(lams: torch.Tensor, level: int, iters: int = 40) -> tuple:
    """Train one profile per λ of the grid (B,) at fixed λ, all from P = −r (c = 0): the coefficients after every
    Gauss–Newton step (iters + 1, B, K) and the funnel after every step (iters + 1, B)."""
    C0 = torch.zeros(len(lams), N_MODES, dtype=lams.dtype, device=lams.device)
    _, _, hist, path = gauss_newton(burgers_residual_fn(level, lams.device), C0, lams, iters, keep_path=True)
    return path, hist


def burgers_funnel(lams: torch.Tensor, level: int, iters: int = 40) -> torch.Tensor:
    """Best residual ‖R‖ at each fixed λ (B,): the funnel."""
    return burgers_funnel_history(lams, level, iters)[1][-1]


def burgers_search(lam0: torch.Tensor, level: int, width: float = 0.03, n_golden: int = 24, n_fit: int = 20) -> dict:
    """Pin down λ near each starting value lam0 (B,): golden-section search on the funnel in [λ0 − w, λ0 + w], the
    profile re-trained (warm start) at every trial λ. level 0: no constraint; level 1: c₃ = 0; level 2: c₃ = c₅ = 0."""
    fn = burgers_residual_fn(level, lam0.device)
    C, _, _ = gauss_newton(fn, torch.zeros(len(lam0), N_MODES, dtype=lam0.dtype, device=lam0.device), lam0, 40)
    lam, C, res, hist = golden_section(fn, lam0 - width, lam0 + width, C, n_golden, n_fit, f"λ search, level {level}")
    C, res, _ = gauss_newton(fn, C, lam, 20)  # converge the profile at the final λ
    return {"lam": lam, "coef": C, "res": res, "lam_hist": hist}


def project_profile(fn, lam: float, r: torch.Tensor, n_modes: int) -> torch.Tensor:
    """Least-squares coefficients c with y P(r) ≈ fn(y) at the points r (P = −r + Σ c_k b_k): (K,)."""
    lam_t = torch.tensor(lam, dtype=r.dtype, device=r.device)
    y = tensors.L_BURGERS * (r ** (-2 * (1 + lam_t)) - 1).sqrt()
    phi, _ = tensors.burgers_features(r, 1 / (2 * (1 + lam_t)), n_modes // tensors.N_FAMILIES)
    return torch.linalg.lstsq(phi, (fn(y) / y + r)[:, None]).solution[:, 0]


# ---------------------------------------------------------------- CCF
CCF_J, CCF_M, CCF_COL = 3, 10, 200


def ccf_residual_fn(device: torch.device):
    y = tensors.ccf_points(CCF_COL, device)
    return lambda A, lam, need_jac=True: equation.ccf_batch(A, lam, y, CCF_J, CCF_M, need_jac)


def ccf_funnel_history(lams: torch.Tensor, iters: int = 40) -> tuple:
    """CCF profiles trained at each fixed λ of the grid (B,): coefficients (B, K), funnel history (iters + 1, B)."""
    A0 = tensors.ccf_initial_coefficients(lams, CCF_J, CCF_M)
    A, _, hist = gauss_newton(ccf_residual_fn(lams.device), A0, lams, iters)
    return A, hist


def joint_polish(A: torch.Tensor, lam: torch.Tensor, iters: int) -> tuple:
    """Gauss–Newton on (a, λ) together for every member; the Jacobian, including the column ∂R/∂λ (λ enters the
    exponents s = jβ + m of the basis), comes from autograd (vmap of jacrev)."""
    y = tensors.ccf_points(CCF_COL, A.device)

    def f(p):  # one member: p = (a₁ … a_K, λ) → R (N + 1,)
        return equation.ccf_residual(p[:-1], p[-1], y, CCF_J, CCF_M)

    jac = torch.func.vmap(torch.func.jacrev(f))

    def residual(P, _, need_jac=True):
        return torch.func.vmap(f)(P), jac(P) if need_jac else None

    P, res, hist = gauss_newton(residual, torch.cat([A, lam[:, None]], 1), lam, iters)
    return P[:, :-1], P[:, -1], res, hist


def ccf_search(
    lam0: torch.Tensor, width: float = 0.06, n_golden: int = 16, n_fit: int = 40, polish: int = 8, warm: int = 60
) -> dict:
    """Golden-section search for λ around lam0 (B,), then a joint Gauss–Newton polish of profile and λ together."""
    fn = ccf_residual_fn(lam0.device)
    A, _, warm_hist = gauss_newton(fn, tensors.ccf_initial_coefficients(lam0, CCF_J, CCF_M), lam0, warm)
    lam, A, res, lam_hist = golden_section(fn, lam0 - width, lam0 + width, A, n_golden, n_fit, "CCF λ search")
    A, res, fit_hist = gauss_newton(fn, A, lam, 3 * n_fit)  # converge the profile at the found λ
    A, lam, res, polish_hist = joint_polish(A, lam, polish)
    y = tensors.ccf_points(CCF_COL, lam0.device)
    omega = torch.stack([equation.ccf_fields(a, lv, y, CCF_J, CCF_M)["omega"] for a, lv in zip(A, lam)], 1)  # (N, B)
    return {
        "lam": lam,
        "coef": A,
        "res": res,
        "lam_history": torch.cat([lam_hist, lam[None]]),
        "residual_history": torch.cat([warm_hist, fit_hist, polish_hist]),
        "omega": omega,
        "y": y,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/blowup_profile_search/plot.py
"""Three acts: the batch of profiles trains and the residual funnels form; golden-section steps pin λ down; then the
found profile blows up, u(x, t) = τ^λ U(x/τ^{1+λ}), with max|u_x| ∝ 1/τ. The CCF panel shows the learned vorticity
profile and its λ against the published values."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import equation

LEVEL_NAMES = ["no constraint", r"$U'''(0)=0$", r"$U'''(0)=U^{(5)}(0)=0$"]
GRAY = "#9ca3af"


def animate(
    lams, funnels, paths, found, exact, taus, x, u, slope, clams, ccf_funnel, ccf, references, path: Path
) -> Path:
    lam_np = lams.cpu().numpy()
    fun = [f.cpu().numpy() for f in funnels]  # (S, B) each
    n_gn = fun[0].shape[0]
    golden = [f["hist"].cpu().numpy() for f in found]
    n_gold = len(golden[0])
    n_blow = len(taus)
    y = torch.linspace(0, 6, 241, dtype=lams.dtype, device=lams.device)

    # profiles of three members of the batch (λ on the grid nearest 1/2, 1/4, 1/6) after every Gauss–Newton step
    members = [int((lams - e).abs().argmin()) for e in exact]
    prof = np.stack(
        [
            np.stack(
                [
                    equation.burgers_profile(paths[lv][s, members[lv]], lams[members[lv]], y)[0].cpu().numpy()
                    for s in range(n_gn)
                ]
            )
            for lv in range(3)
        ]
    )  # (3, S, Y)
    batch_idx = list(range(0, len(lam_np), max(1, len(lam_np) // 8)))
    batch = np.stack(
        [
            np.stack([equation.burgers_profile(paths[0][s, b], lams[b], y)[0].cpu().numpy() for b in batch_idx])
            for s in range(n_gn)
        ]
    )  # (S, nb, Y)
    exact_prof = [equation.burgers_exact(y, i)[0].cpu().numpy() for i in range(3)]
    y_np = y.cpu().numpy()

    fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.8))
    (ax_p, ax_f, ax_l), (ax_u, ax_s, ax_c) = axes

    # --- profiles
    batch_lines = [ax_p.plot(y_np, batch[0, k], color=GRAY, lw=0.6, alpha=0.6)[0] for k in range(len(batch_idx))]
    for i in range(3):
        ax_p.plot(y_np, exact_prof[i], color=ACCENT[i], lw=1.0, ls="--")
    prof_lines = [
        ax_p.plot(
            y_np, prof[i, 0], color=ACCENT[i], lw=2, label=rf"$\lambda={lam_np[members[i]]:.3f}$, {LEVEL_NAMES[i]}"
        )[0]
        for i in range(3)
    ]
    ax_p.plot([], [], color="#374151", ls="--", lw=1, label=r"exact $y=-U-U^{2i+3}$")
    ax_p.set_xlim(0, 6)
    ax_p.set_ylim(-2.2, 0.3)
    ax_p.set_xlabel("$y$")
    ax_p.set_ylabel("$U(y)$")
    ax_p.legend(fontsize=6.5, loc="lower left")
    title_p = ax_p.set_title("", fontsize=9)

    # --- funnels
    fun_lines = [ax_f.semilogy(lam_np, fun[i][0], color=ACCENT[i], lw=1.6, label=LEVEL_NAMES[i])[0] for i in range(3)]
    for e, name in zip(exact, ["1/2", "1/4", "1/6"]):
        ax_f.axvline(e, color=GRAY, ls=":", lw=1)
        ax_f.text(e, 2e-2, name, fontsize=7, ha="center", color="#4b5563")
    lo = min(f[-1].min() for f in fun)
    ax_f.set_ylim(max(lo, 1e-17) / 3, max(f.max() for f in fun) * 3)
    ax_f.set_xlabel(r"$\lambda$ (fixed while $U$ trains)")
    ax_f.set_ylabel(r"best residual $\|R\|$")
    ax_f.legend(fontsize=7, loc="lower right")
    title_f = ax_f.set_title("", fontsize=9)
    marks = [ax_f.plot([], [], "o", color=ACCENT[i], ms=6, mec="white")[0] for i in range(3)]

    # --- λ search
    steps_g = np.arange(n_gold)
    err = [np.abs(golden[i] - exact[i]).clip(1e-16) for i in range(3)]
    err_lines = [
        ax_l.semilogy(
            [], [], "o-", color=ACCENT[i], ms=3, lw=1.3, label=rf"level {i}: $\lambda\to${found[i]['lam']:.5f}"
        )[0]
        for i in range(3)
    ]
    ax_l.set_xlim(0, n_gold - 1)
    ax_l.set_ylim(1e-8, 0.1)
    ax_l.set_xlabel("golden-section step")
    ax_l.set_ylabel(r"$|\lambda - 1/(2i+2)|$")
    ax_l.legend(fontsize=7, loc="lower left")
    ax_l.set_title(r"pinning $\lambda$ down (profile retrained at each trial $\lambda$)", fontsize=9)

    # --- blow-up
    x_np, u_np, tau_np, s_np = x.cpu().numpy(), u.cpu().numpy(), taus.cpu().numpy(), slope.cpu().numpy()
    ghosts = [ax_u.plot(x_np, u_np[0], color=ACCENT[0], lw=0.8, alpha=0.25)[0] for _ in range(3)]
    u_line = ax_u.plot(x_np, u_np[0], color=ACCENT[0], lw=2)[0]
    ax_u.set_xlim(-1, 1)
    ax_u.set_ylim(-1.1, 1.1)
    ax_u.set_xlabel("$x$")
    ax_u.set_ylabel("$u(x,t)$")
    title_u = ax_u.set_title("", fontsize=9)

    ax_s.loglog(tau_np, s_np, color=ACCENT[0], lw=2, label=r"$\max|u_x|$ from the learned $U$")
    ax_s.loglog(tau_np, 1 / tau_np, color="#374151", ls="--", lw=1, label=r"$1/(T-t)$")
    s_mark = ax_s.plot([tau_np[0]], [s_np[0]], "o", color=ACCENT[1], ms=7)[0]
    ax_s.invert_xaxis()
    ax_s.set_xlabel(r"$T-t$")
    ax_s.set_ylabel(r"$\max|u_x|$")
    ax_s.legend(fontsize=7, loc="upper left")
    ax_s.set_title(r"gradient blow-up: $\max|u_x| = |U'(0)|/(T-t)$", fontsize=9)

    # --- CCF
    yc = ccf["y"].cpu().numpy()
    om = ccf["omega"][:, 0].cpu().numpy()
    ax_c.semilogx(yc, om, color=ACCENT[3], lw=2, label=r"learned $\Omega(y)$, odd")
    ax_c.set_xlim(yc[1], yc[-1])
    ax_c.set_ylim(-0.75, 0.05)
    ax_c.set_xlabel("$y$ (log scale; $\\Omega(-y) = -\\Omega(y)$)")
    ax_c.set_ylabel(r"$\Omega$")
    lam_c = ccf["lam"][0].item()
    ax_c.set_title(rf"CCF stable profile: $\lambda$ = {lam_c:.4f}, $\|R\|$ = {ccf['res'][0]:.0e}", fontsize=9)
    ins = ax_c.inset_axes([0.56, 0.5, 0.41, 0.42])
    ins.semilogy(clams.cpu().numpy(), ccf_funnel.cpu().numpy(), color=ACCENT[3], lw=1.3)
    for k, (name, val) in enumerate(references.items()):
        ins.axvline(val, color=ACCENT[4 + k], ls=":", lw=1.2)
        ax_c.plot([], [], color=ACCENT[4 + k], ls=":", lw=1.2, label=name)
    ins.plot([lam_c], [ccf["res"][0].item()], "o", color=ACCENT[3], ms=5)
    ins.tick_params(labelsize=6)
    ins.set_xlabel(r"$\lambda$", fontsize=7, labelpad=0)
    ins.set_title("CCF funnel vs published", fontsize=7)
    ax_c.legend(fontsize=6.5, loc="lower right")

    def update(frame: int) -> None:
        s = min(frame, n_gn - 1)
        for k, line in enumerate(batch_lines):
            line.set_ydata(batch[s, k])
        for i in range(3):
            prof_lines[i].set_ydata(prof[i, s])
            fun_lines[i].set_ydata(fun[i][s])
        title_p.set_text(f"batch of profiles, Gauss–Newton step {s}/{n_gn - 1}")
        title_f.set_text(f"residual funnels, step {s}: smooth profiles only at $\\lambda = 1/(2i+2)$")
        g = frame - n_gn
        if g >= 0:
            gs = min(g, n_gold - 1)
            for i in range(3):
                err_lines[i].set_data(steps_g[: gs + 1], err[i][: gs + 1])
                j = int(np.abs(lam_np - golden[i][gs]).argmin())
                marks[i].set_data([golden[i][gs]], [fun[i][-1][j]])
        b = frame - n_gn - n_gold
        k = int(np.clip(b, 0, n_blow - 1))
        u_line.set_ydata(u_np[k])
        for m, ghost in enumerate(ghosts):
            ghost.set_ydata(u_np[max(0, k - 3 * (m + 1))])
        s_mark.set_data([tau_np[k]], [s_np[k]])
        title_u.set_text(rf"blow-up with the learned $\lambda$ = {found[0]['lam']:.4f}: $T-t$ = {tau_np[k]:.1e}")

    fig.tight_layout()
    n_total = n_gn + n_gold + n_blow
    return save_gif(fig, update, frame_indices(n_total, 40), path, fps=4)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_lambda': 61,
    'gn_steps': 40,
    'ccf_lambda0': 1.2
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.blowup_profile_search")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.blowup_profile_search.main")
problem_dir = Path("problems/fluids/blowup_profile_search")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
